In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ecommerce-medallion-pipeline") \
    .getOrCreate()

print("Done")

Done


In [0]:
BASE_PATH = "/Volumes/workspace/default/ecommerce_medallion_pipeline"

RAW_LAYER_PATH = f"{BASE_PATH}/raw_data/"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoint/"
BRONZE_LAYER_PATH = f"{BASE_PATH}/bronze/"
SILVER_LAYER_PATH = f"{BASE_PATH}/silver/"
GOLD_LAYER_PATH = f"{BASE_PATH}/gold/"

# ------------------------------------------------------------
# One config entry per table: (table_name, delimiter)
# customers.csv uses ";" (from the generator); everything else uses ","
# ------------------------------------------------------------
TABLES = [
    ("departments", ","),
    ("employees", ","),
    ("categories", ","),
    ("products", ","),
    ("suppliers", ","),
    ("product_suppliers", ","),
    ("customers", ";"),
    ("shippers", ","),
    ("orders", ","),
    ("order_details", ","),
    ("payments", ","),
    ("shipments", ","),
    ("orders_cdc_incremental", ","),
]

# each table needs its OWN raw folder, e.g. ./raw_data/customers/customers.csv,
# not one shared folder with all CSVs mixed together 
# bec, Auto Loader watches a folder and each stream below is scoped to just its table's folder.

active_queries = []

for table_name, delimiter in TABLES:
    raw_path = f"{RAW_LAYER_PATH}{table_name}/"
    bronze_path = f"{BRONZE_LAYER_PATH}{table_name}"
    schema_location = f"{CHECKPOINT_PATH}{table_name}_schema"
    checkpoint_location = f"{CHECKPOINT_PATH}{table_name}_data"

    # cloudFiles = that we use aoutolader 
    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("sep", delimiter)
        .option("escape", '"')   # we added this bec we fond that the products file has a column contains json format
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaEvolutionMode", "rescue")  # captures unexpected columns into _rescued_data
        .load(raw_path)
    )

    query = (
        df.writeStream.format("delta")
        .option("checkpointLocation", checkpoint_location)
        .option("mergeSchema", "true")  # this is the schema eveloation let new columns from _rescued_data evolve into the Delta table
        .outputMode("append")
        .trigger(availableNow=True)  # process what's new today, then stop -- not a 24/7 stream
        .start(bronze_path)
    )

    active_queries.append((table_name, query))
    print(f"{table_name} -> Bronze stream started")

# Wait for every table's ingestion to finish before moving on to Silver.
# Since each is a separate stream, one table's schema hiccup won't silently
# block the others -- we still want to know if any of them failed.
failures = []
for table_name, query in active_queries:
    try:
        query.awaitTermination()
        print(f"{table_name} -> Bronze Done ({query.lastProgress.get('numInputRows', 'n/a') if query.lastProgress else 0} rows this run)")
    except Exception as e:
        failures.append((table_name, str(e)))
        print(f"{table_name} -> FAILED: {e}")

if failures:
    raise RuntimeError(f"Bronze ingestion failed for: {[t for t, _ in failures]}")

print("\nAll tables -> Bronze layer complete.")

departments -> Bronze stream started
employees -> Bronze stream started
categories -> Bronze stream started
products -> Bronze stream started
suppliers -> Bronze stream started
product_suppliers -> Bronze stream started
customers -> Bronze stream started
shippers -> Bronze stream started
orders -> Bronze stream started
order_details -> Bronze stream started
payments -> Bronze stream started
shipments -> Bronze stream started
orders_cdc_incremental -> Bronze stream started
departments -> Bronze Done (None rows this run)
employees -> Bronze Done (None rows this run)
categories -> Bronze Done (None rows this run)
products -> Bronze Done (None rows this run)
suppliers -> Bronze Done (None rows this run)
product_suppliers -> Bronze Done (None rows this run)
customers -> Bronze Done (None rows this run)
shippers -> Bronze Done (None rows this run)
orders -> Bronze Done (None rows this run)
order_details -> Bronze Done (None rows this run)
payments -> Bronze Done (None rows this run)
shipment